# Table statistics


In [ ]:
%%capture
%pip install matplotlib

PyProBE filter objects inherit from `Table`, so statistical reductions can be applied directly to any filtered slice of data. This example uses the same sample Neware dataset as the other notebooks and shows the reducers that return a single value, followed by the grouped `summary()` method.


In [ ]:
import matplotlib.pyplot as plt
import polars as pl

import pyprobe
from pyprobe.filters import Procedure

%matplotlib inline

In [ ]:
cell = pyprobe.Cell()

data_directory = "../../../tests/sample_data/neware"
procedure = Procedure.load(
    data_directory + "/sample_data_neware.bdx.parquet",
    readme_path=data_directory + "/README.yaml",
)
cell.add_procedure("Sample", procedure)

We will work with the first discharge in the `Break-in Cycles` experiment:


In [ ]:
break_in = cell.procedure["Sample"].experiment("Break-in Cycles")
first_cycle = break_in.cycle(0)
first_discharge = first_cycle.discharge(0)

first_discharge.data.head()

Reducers such as `first()`, `last()`, `delta()`, `range()`, `mean()`, `minimum()`, and `maximum()` all return a single-row `Table`. The `item()` method is convenient when you want the result as a scalar.


In [ ]:
statistics = pl.DataFrame(
    {
        "Statistic": [
            "first Voltage / V",
            "last Voltage / V",
            "delta Net Capacity / Ah",
            "range Voltage / V",
            "mean Current / mA",
            "min Voltage / V",
            "max Voltage / V",
        ],
        "Value": [
            first_discharge.first("Voltage / V").item(),
            first_discharge.last("Voltage / V").item(),
            first_discharge.delta("Net Capacity / Ah").item(),
            first_discharge.range("Voltage / V").item(),
            first_discharge.mean("Current / mA").item(),
            first_discharge.minimum("Voltage / V").item(),
            first_discharge.maximum("Voltage / V").item(),
        ],
    }
).with_columns(pl.col("Value").round(4))

statistics

For a single discharge, `range("Net Capacity / Ah")` returns the span between the minimum and maximum capacity values in that slice. Plotting capacity together with voltage makes it easier to see both the size of the capacity window and where it occurs during the discharge.


In [ ]:
capacity_range_mAh = first_discharge.range("Net Capacity / mAh").item()
minimum_capacity = first_discharge.minimum("Net Capacity / mAh").item()
maximum_capacity = first_discharge.maximum("Net Capacity / mAh").item()

fig, ax = plt.subplots(figsize=(8, 4))
first_discharge.plot(
    x="Test Time / hr",
    y="Net Capacity / mAh",
    ax=ax,
    color="C1",
    label="Capacity",
    legend=False,
)
ax.axhline(minimum_capacity, color="C1", linestyle="--", label="Minimum capacity")
ax.axhline(maximum_capacity, color="C2", linestyle="--", label="Maximum capacity")
ax.set_ylabel("Net Capacity / mAh")

ax2 = ax.twinx()
first_discharge.plot(
    x="Test Time / hr",
    y="Voltage / V",
    ax=ax2,
    color="C0",
    label="Voltage",
    legend=False,
)
ax2.set_ylabel("Voltage / V")
ax.set_title(f"Capacity range over the first discharge: {capacity_range_mAh:.2f} mAh")
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(
    lines + lines2,
    labels + labels2,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=2,
)
fig.tight_layout()

The grouped `summary()` method applies the full statistical suite at once. By default it groups by `Step Count / 1`, which makes it useful for comparing each step within a cycle.


In [ ]:
step_summary = first_cycle.summary("Net Capacity / Ah", "Voltage / V")

step_summary.data.select(
    "Step Count / 1",
    "Step ID",
    "delta Net Capacity / Ah",
    "range Net Capacity / Ah",
    "mean Voltage / V",
    "first Voltage / V",
    "last Voltage / V",
).with_columns(
    pl.exclude("Step Count / 1", "Step ID").round(4),
)